In [1]:
import numpy as np
import networkx as nx
import math

def generate_dataset(n=20, m=16, lambda_=3, mu_=3):
    """
    Generates a synthetic reviewer-to-paper similarity matrix using a fixed random seed.

    Parameters:
    -----------
    n       : int, Total number of reviewers available.
    m       : int, Total number of papers submitted.
    lambda_ : int, Reviewers required per paper (demand).
    mu_     : int, Max papers a single reviewer can review (capacity).
    """
    np.random.seed(98)  # Ensures deterministic matrix generation for reproducible benchmarks

    # Generate similarities uniformly distributed between 0.2 and 0.8
    S = np.random.uniform(0.05, 0.95, (n, m))

    # Note: Bottleneck constraints are commented out below but can be un-commented
    # to test how algorithms perform under extreme fairness constraints.
    S[:, 0] = np.random.uniform(0.05, 0.2, n)
    S[0, 0] = 0.95
    S[1, 0] = 0.85

    return S, n, m, lambda_, mu_

def peer_review_4_all_simplified(S, lambda_, mu_):
    """
    An iterative Max-Flow heuristic based on the PeerReview4All (PR4A) algorithm.

    Instead of optimizing global welfare at once, it iteratively builds a flow network,
    identifies the paper that is currently "worst-off" (lowest cumulative similarity score),
    permanently locks in its reviewers, and repeats until all papers are satisfied.
    """
    n, m = S.shape
    assignment = {j: [] for j in range(m)}        # Final assignments mapping: {paper_id: [reviewer_ids]}
    reviewer_caps = {i: mu_ for i in range(n)}     # Dynamic capacities for reviewers
    papers_needed = {j: lambda_ for j in range(m)}  # Dynamic remaining demands for papers

    # Flatten the similarity matrix into a list of tuples: (similarity_score, reviewer_id, paper_id)
    sims = []
    for i in range(n):
        for j in range(m):
            sims.append((S[i, j], i, j))

    # Sort all pairs by similarity in descending order.
    # This lets us incrementally add the highest-quality edges into our flow network first.
    sims.sort(key=lambda x: x[0], reverse=True)

    def check_flow(k_target, active_papers):
        """
        Subroutine that evaluates if a valid assignment exists when papers demand
        'k_target' reviewers, using only the highest possible similarity edges.
        """
        G = nx.DiGraph()
        G.add_node('Source')
        G.add_node('Sink')

        # Connect Source to available Reviewers with their remaining review capacity
        for i in range(n):
            if reviewer_caps[i] > 0:
                G.add_edge('Source', f'R_{i}', capacity=reviewer_caps[i])

        # Connect active Papers to Sink with a capacity of the requested target quota
        for j in active_papers:
            needed = min(papers_needed[j], k_target)
            G.add_edge(f'P_{j}', 'Sink', capacity=needed)

        # Iterating through sorted pairs, sequentially adding reviewer-paper edges to the graph.
        # We stop as soon as the maximum network flow equals the required target flow.
        # This guarantees we find the maximum possible "similarity threshold" to satisfy the constraints.
        for sim, i, j in sims:
            if j in active_papers and reviewer_caps[i] > 0 and i not in assignment[j]:
                G.add_edge(f'R_{i}', f'P_{j}', capacity=1)

            # Compute max flow through the constructed network using NetworkX
            flow_val, flow_dict = nx.maximum_flow(G, 'Source', 'Sink')
            target_flow = sum(min(papers_needed[p], k_target) for p in active_papers)

            # If the current edge set can fully satisfy the flow demand, return this state
            if flow_val == target_flow:
                return flow_dict, sim

        return None, -1

    active_papers = list(range(m))
    while active_papers:
        best_flow = None
        best_k = 1

        # Search for the highest feasible review quota (k) we can fulfill
        for k in range(1, lambda_ + 1):
            flow_dict, min_sim = check_flow(k, active_papers)
            if flow_dict is not None:
                best_flow = flow_dict
                best_k = k

        if not best_flow:
            raise ValueError("No feasible flow found. Constraints too tight.")

        # Analyze the successful flow to build a temporary candidate assignment
        temp_scores = {p: 0 for p in active_papers}
        temp_assignment = {p: [] for p in active_papers}
        for i in range(n):
            if f'R_{i}' in best_flow:
                for dest, flow in best_flow[f'R_{i}'].items():
                    # If an edge carries flow, it means reviewer i is assigned to paper p in this round
                    if flow > 0 and dest.startswith('P_'):
                        p = int(dest.split('_')[1])
                        temp_scores[p] += S[i, p]   # Track cumulative similarity score for the paper
                        temp_assignment[p].append(i)

        # --- THE MAX-MIN FAIRNESS CORE STEP ---
        # Identify the "worst-off" paper (the one with the lowest total reviewer similarity)
        worst_paper = min(active_papers, key=lambda p: temp_scores[p])

        # Permanently lock in the assignments for this specific worst-off paper
        for rev in temp_assignment[worst_paper]:
            assignment[worst_paper].append(rev)
            reviewer_caps[rev] -= 1       # Reduce reviewer's remaining availability
            papers_needed[worst_paper] -= 1 # Reduce paper's remaining demand

        # If the worst-off paper has met its full coverage constraint (lambda_), remove it from active processing
        if papers_needed[worst_paper] == 0:
            active_papers.remove(worst_paper)

    return assignment

def min_cost_flow_nsw(S, lambda_, mu_, threshold=0.7):
    """
    Solves the Binary Nash Social Welfare (NSW) optimization problem using Min-Cost Flow.

    Nash Social Welfare maximizes the geometric mean of paper scores. By taking the logarithm,
    the product optimization turns into a summation optimization: Maximize Sum( log(scores) ).

    To implement this in a Min-Cost Flow network (which minimizes costs), we assign negative
    weights to the utility gains. Diminishing returns are modeled by splitting each paper
    into individual reviewer slots, where each subsequent slot carries a smaller negative cost.
    """
    n, m = S.shape

    # Establish a binary threshold matrix: 1 if similarity is good (>= threshold), else 0
    B = (S >= threshold).astype(int)

    G = nx.DiGraph()
    G.add_node('Source')
    G.add_node('Sink')

    # Source connects to Reviewers. Capacity = mu_ (max reviews a person can perform), Cost = 0
    for i in range(n):
        G.add_edge('Source', f'R_{i}', capacity=mu_, weight=0)

    # Build the incremental logarithmic gadgets for the Papers
    for j in range(m):
        for k in range(1, lambda_ + 1):
            # Calculate the marginal utility of adding the k-th "good" reviewer using log differences
            marginal_utility = math.log(k + 1) - math.log(k)

            # Scale and invert the utility into a negative integer cost (since NetworkX minimizes cost)
            cost = int(-1000 * marginal_utility)

            # Create a separate tracking node for each unique slot (k) of Paper j
            node_name = f'P_{j}_k{k}'

            # Main paper node routes flow into its slot-specific node
            G.add_edge(f'P_{j}', node_name, capacity=1, weight=0)
            # Slot-specific node connects to the Sink, carrying the logarithmic reward (negative cost)
            G.add_edge(node_name, 'Sink', capacity=1, weight=cost)

    # Establish reviewer-to-paper assignment edges
    for i in range(n):
        for j in range(m):
            if B[i, j] == 1:
                # High-quality matches are free to route (cost = 0)
                G.add_edge(f'R_{i}', f'P_{j}', capacity=1, weight=0)
            else:
                # Low-quality matches carry a massive cost penalty (5000).
                # The solver will only use this edge if it's mathematically impossible to satisfy demand otherwise.
                G.add_edge(f'R_{i}', f'P_{j}', capacity=1, weight=5000)

    # Define strict supply and demand requirements across the closed flow system
    G.nodes['Source']['demand'] = - (m * lambda_)  # Source supplies total required reviews
    G.nodes['Sink']['demand'] = (m * lambda_)     # Sink absorbs total required reviews

    # Execute NetworkX's Network Simplex algorithm to find the optimal Min-Cost Flow solution
    flow_cost, flow_dict = nx.network_simplex(G)

    # Extract assignments from the computed flow dictionary
    assignment = {j: [] for j in range(m)}
    for i in range(n):
        for dest, flow in flow_dict[f'R_{i}'].items():
            # If flow travels from Reviewer i to Paper p, log the match
            if flow > 0 and dest.startswith('P_'):
                p = int(dest.split('_')[1])
                assignment[p].append(i)

    return assignment

def evaluate_metrics(S, assignment):
    """
    Computes benchmark metrics to evaluate assignment matrices based on continuous similarities.

    Returns:
    --------
    nsw              : float, Geometric mean of paper profiles (penalizes zero/poor matching distribution).
    fairness         : float, Minimum individual cumulative valuation across all papers (Max-Min Floor).
    sorted(valuations): list, Ascending list of total profile scores per paper.
    """
    m = S.shape[1]
    valuations = []

    # Calculate real continuous utility score for each paper based on its assigned reviewers
    for j in range(m):
        score = sum(S[i, j] for i in assignment[j])
        valuations.append(score)

    # Max-Min Fairness is defined as the score of the most disadvantaged paper
    fairness = min(valuations)

    # Nash Social Welfare (NSW) is the geometric mean of valuations.
    # Uses max(v, 1e-4) as a numerical guard rail against log(0) calculation crashes.
    nsw = np.prod([max(v, 1e-4) for v in valuations]) ** (1/m)

    return nsw, fairness, sorted(valuations)


# ==========================================
# RUN BENCHMARK EXPERIMENT
# ==========================================
S, n, m, lambda_, mu_ = generate_dataset()

# 1. Run both assignment techniques
assign_pr4a = peer_review_4_all_simplified(S, lambda_, mu_)
assign_nsw = min_cost_flow_nsw(S, lambda_, mu_, threshold=0.4)

# 2. Extract and format metrics
nsw_a, fair_a, vals_a = evaluate_metrics(S, assign_pr4a)
nsw_b, fair_b, vals_b = evaluate_metrics(S, assign_nsw)

# 3. Output Results Panel
print("=== PeerReview4All (Algorithm A) ===")
print(f"Fairness (Max-Min) : {fair_a:.4f}")
print(f"Nash Social Welfare: {nsw_a:.4f}")
print(f"Sorted Valuations  : {[round(v, 3) for v in vals_a]}\n")

print("=== Binary NSW via Min-Cost Flow (Algorithm B) ===")
print(f"Fairness (Max-Min) : {fair_b:.4f}")
print(f"Nash Social Welfare: {nsw_b:.4f}")
print(f"Sorted Valuations  : {[round(v, 3) for v in vals_b]}")

=== PeerReview4All (Algorithm A) ===
Fairness (Max-Min) : 0.9695
Nash Social Welfare: 1.9772
Sorted Valuations  : [np.float64(0.97), np.float64(1.452), np.float64(1.511), np.float64(1.591), np.float64(1.661), np.float64(1.726), np.float64(1.883), np.float64(1.926), np.float64(1.999), np.float64(2.299), np.float64(2.399), np.float64(2.638), np.float64(2.677), np.float64(2.694), np.float64(2.712), np.float64(2.726)]

=== Binary NSW via Min-Cost Flow (Algorithm B) ===
Fairness (Max-Min) : 1.3217
Nash Social Welfare: 1.9682
Sorted Valuations  : [np.float64(1.322), np.float64(1.704), np.float64(1.743), np.float64(1.762), np.float64(1.796), np.float64(1.896), np.float64(1.901), np.float64(1.926), np.float64(1.969), np.float64(1.995), np.float64(2.135), np.float64(2.136), np.float64(2.307), np.float64(2.374), np.float64(2.445), np.float64(2.452)]
